# Cloud HAND-FIM: Python SDK submission and monitoring

This notebook imports the **generated OpenAPI Python SDK** directly to call a Cloud Run API. No CLI subprocesses are used. HydroShare authentication and an S3 output bucket are required. Jobs incur cloud costs. Run cells in order; never rerun submission to recover from a monitoring failure. Use `RESUME_JOB_ID` instead.

Browser approval delivers a session token directly to the SDK without copy-and-paste. Do not print token variables, client objects, or authentication responses. See the [deployment instructions](../readme.md). Select a Python 3.10+ kernel.

In [ ]:
%pip install httpx

from pathlib import Path
import importlib
import site
import sysconfig

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/fimbox_sdk').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the FIMBox checkout, or set ROOT to its absolute path.')
%pip install -e "$ROOT"

# Load editable-install .pth files created after this kernel started.
for package_dir in {sysconfig.get_path('purelib'), sysconfig.get_path('platlib')}:
    site.addsitedir(package_dir)
importlib.invalidate_caches()

from fimbox_sdk import Client, AuthenticatedClient
from fimbox_sdk.api.authentication import authentication_config_auth_config_get, current_session_auth_session_get
from fimbox_sdk.api.default import submit_hand_fim_jobs_hand_fim_post, get_job_status_jobs_job_id_get, get_job_logs_jobs_job_id_logs_get
from fimbox_sdk.models import AuthenticationConfig, HandFimJobRequest, JobLogs, JobStatus, JobSubmitted, Reach, Scenario, StoredHandFimResult


## Configure the destination
Get the HTTPS endpoint with `terraform -chdir=terraform output -raw api_url` in a terminal. Use your HydroShare bucket, not the Google Artifact Registry repository. Leave `RESUME_JOB_ID` empty only for a new job. `MAX_PROCS=1` limits scenario concurrency; Cloud Run scratch files still consume worker memory.

In [ ]:
import os
from urllib.parse import urlsplit

API_URL = os.environ.get('FIMBOX_API_URL', 'https://fimbox-dev-api-307400264918.us-central1.run.app').rstrip('/')
OUTPUT_BUCKET = 'resource'
OUTPUT_PREFIX = 'ad1ae21dec7648efa78d5015c73728d4/data/contents/fimbox_output/'
MAX_PROCS = 1
RESUME_JOB_ID = ''
POLL_SECONDS = 10
WAIT_SECONDS = 1800

origin = urlsplit(API_URL)
if origin.scheme != 'https' or not origin.hostname or origin.username or origin.password or origin.query or origin.fragment or 'YOUR-' in API_URL:
    raise ValueError('Set API_URL to the actual HTTPS API endpoint, without credentials or query parameters.')
if OUTPUT_BUCKET == 'YOUR_HYDROSHARE_BUCKET' or not OUTPUT_BUCKET:
    raise ValueError('Set OUTPUT_BUCKET to your HydroShare S3 bucket.')


## Authenticate
Run the next cell, click its login link, confirm you initiated this login, and authorize with HydroShare. The cell polls for up to ten minutes and receives the FIMBox session token automatically, including with remote kernels. An existing FIMBOX_SESSION_TOKEN environment value takes precedence. Tokens remain in memory. If login expires, is denied, or loses its connection, rerun this cell to start a new transaction.

In [ ]:
from fimbox_login import begin_login
import html
import httpx
from IPython.display import HTML, display

def call_api(operation, *, client, expected_status=200, expected_type=None, **kwargs):
    try:
        response = operation.sync_detailed(client=client, **kwargs)
    except httpx.RequestError:
        raise RuntimeError('API connection failed. If submitting, inspect your jobs before attempting another submission.') from None
    except (ValueError, KeyError, TypeError, AttributeError):
        raise RuntimeError('API response does not match the generated SDK contract.') from None
    if response.status_code != expected_status:
        raise RuntimeError(f'API request failed (HTTP {response.status_code}). Check endpoint, session expiry, bucket access, and server logs.')
    if expected_type is not None and not isinstance(response.parsed, expected_type):
        raise RuntimeError('API returned an unexpected response model.')
    return response.parsed

session_token = ''
with Client(base_url=API_URL, timeout=httpx.Timeout(30), follow_redirects=False) as public_client:
    auth_config = call_api(authentication_config_auth_config_get, client=public_client, expected_type=AuthenticationConfig)
    if auth_config.authentication_enabled is not True:
        raise RuntimeError('Expected a reachable, authentication-enabled cloud API.')
    session_token = os.environ.get('FIMBOX_SESSION_TOKEN', '').strip()
    if not session_token:
        browser_login = begin_login(public_client)
        display(HTML(f'<a href="{html.escape(browser_login.authorization_url, quote=True)}" target="_blank" rel="noopener noreferrer">Authorize FIMBox</a>'))
        print('Open the link and authorize. Waiting for browser approval...')
        session_token = browser_login.wait()
        del browser_login
print('Session received. Run the Connect cell below.')


### Connect using the browser-approved session

Run this cell after authentication completes. It validates the received session. Never print the token or client object. Clear outputs before sharing. Redeploy the cloud API with polling endpoints before using this notebook.

In [ ]:
if not session_token.strip():
    raise ValueError('Run the authentication cell and authorize in the browser first.')

previous_client = globals().get('api_client')
if isinstance(previous_client, AuthenticatedClient):
    previous_client.get_httpx_client().close()
del previous_client
api_client = AuthenticatedClient(base_url=API_URL, token=session_token.strip(),
                                 timeout=httpx.Timeout(30), follow_redirects=False)
_ = call_api(current_session_auth_session_get, client=api_client)
print('Authenticated to the cloud API.')


In [ ]:
from uuid import UUID

scenarios = [Scenario(scenario_id='single-reach-scenario', reaches=[
    Reach(huc='03020201', reach_id='8780625', streamflow=7)
])]

submitted = call_api(submit_hand_fim_jobs_hand_fim_post, client=api_client,
                        expected_status=202, expected_type=JobSubmitted,
                        body=HandFimJobRequest(scenarios=scenarios, output_bucket=OUTPUT_BUCKET,
                                               output_prefix=OUTPUT_PREFIX, max_procs=MAX_PROCS))
job_id = submitted.job_id


In [ ]:
import time

deadline = time.monotonic() + WAIT_SECONDS
log_offset = 0
while True:
    log_response = call_api(get_job_logs_jobs_job_id_logs_get, client=api_client,
                            expected_type=JobLogs, job_id=UUID(job_id), offset=log_offset)
    log_offset = log_response.next_offset
    state = log_response.status
    print(f'{time.strftime("%H:%M:%S")}  {state}', flush=True)
    if log_response.logs:
        print(log_response.logs, end='', flush=True)
    if state == 'success':
        status_response = call_api(get_job_status_jobs_job_id_get, client=api_client,
                                   expected_type=JobStatus, job_id=UUID(job_id))
        final_result = status_response.result
        if not isinstance(final_result, StoredHandFimResult):
            raise RuntimeError('Successful job has no typed output result.')
        break
    if state in {'failure', 'revoked'}:
        raise RuntimeError(f'Job ended with {state}; do not automatically resubmit.')
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError('Stopped waiting. The cloud job continues; rerun this monitoring cell, not submission.')
    time.sleep(min(POLL_SECONDS, remaining))

for artifact in final_result.files:
    display_path = artifact.path.removeprefix(f'{job_id}-')
    print(display_path, artifact.size_bytes, 'bytes')
